# Data Preparation & Cleaning Pipeline
Script ini dibuat untuk membersihkan dan menyiapkan dataset historis untuk kompetisi time series forecasting.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### 1. Load Datasets

In [ ]:
df_klaim = pd.read_csv('Data_Klaim.csv')
df_polis = pd.read_csv('Data_Polis.csv')

### 2. Pembersihan Logika Finansial (Data_Klaim.csv)

In [ ]:
# Hapus seluruh baris di mana Nominal Klaim Yang Disetujui == 0
df_klaim = df_klaim[df_klaim['Nominal Klaim Yang Disetujui'] != 0].copy()

# Jika Nominal Biaya RS Yang Terjadi == 0 TETAPI Nominal Klaim Yang Disetujui > 0, samakan nilainya
mask_biaya_rs_0 = (df_klaim['Nominal Biaya RS Yang Terjadi'] == 0) & (df_klaim['Nominal Klaim Yang Disetujui'] > 0)
df_klaim.loc[mask_biaya_rs_0, 'Nominal Biaya RS Yang Terjadi'] = df_klaim.loc[mask_biaya_rs_0, 'Nominal Klaim Yang Disetujui']

# INISIATIF: Pastikan tidak ada nilai negatif pada kolom finansial klaim akibat anomali data
df_klaim['Nominal Klaim Yang Disetujui'] = df_klaim['Nominal Klaim Yang Disetujui'].clip(lower=0)
df_klaim['Nominal Biaya RS Yang Terjadi'] = df_klaim['Nominal Biaya RS Yang Terjadi'].clip(lower=0)

### 3. Penanganan Missing Values

In [ ]:
cols_to_fill = ['Inpatient/Outpatient', 'ICD Diagnosis', 'ICD Description', 'Lokasi RS']
for col in cols_to_fill:
    if col in df_klaim.columns:
        df_klaim[col] = df_klaim[col].fillna('Unknown')

### 4. Standarisasi Format Waktu

In [ ]:
# Data_Klaim.csv
date_cols_klaim = ['Tanggal Pembayaran Klaim', 'Tanggal Pasien Masuk RS', 'Tanggal Pasien Keluar RS']
for col in date_cols_klaim:
    if col in df_klaim.columns:
        df_klaim[col] = pd.to_datetime(df_klaim[col], errors='coerce')

# Data_Polis.csv (Format: YYYYMMDD -> datetime)
date_cols_polis = ['Tanggal Lahir', 'Tanggal Efektif Polis']
for col in date_cols_polis:
    if col in df_polis.columns:
        # Dikonversi menjadi string, di-padding jika ada yang kurang dari 8 karakter, lalu ubah ke datetime
        df_polis[col] = pd.to_datetime(df_polis[col].astype(str).str.zfill(8), format='%Y%m%d', errors='coerce')

### 5. Penanganan Outliers (Capping)

In [ ]:
# Hitung persentil ke-99
p99 = df_klaim['Nominal Klaim Yang Disetujui'].quantile(0.99)

# Buat kolom Nominal_Klaim_Capped dan batasi nilainya menggunakan np.where
df_klaim['Nominal_Klaim_Capped'] = np.where(df_klaim['Nominal Klaim Yang Disetujui'] > p99, p99, df_klaim['Nominal Klaim Yang Disetujui'])

### 6. Penggabungan Data & Feature Engineering Tambahan

In [ ]:
# Left Join Data_Klaim ke Data_Polis menggunakan kunci 'Nomor Polis'
df_master = df_klaim.merge(df_polis, on='Nomor Polis', how='left')

# INISIATIF: Membuat fitur Durasi_Perawatan_Hari jika waktu valid
if 'Tanggal Pasien Masuk RS' in df_master.columns and 'Tanggal Pasien Keluar RS' in df_master.columns:
    df_master['Durasi_Perawatan_Hari'] = (df_master['Tanggal Pasien Keluar RS'] - df_master['Tanggal Pasien Masuk RS']).dt.days
    # Imputasi jika hari negatif akibat salah input
    df_master['Durasi_Perawatan_Hari'] = df_master['Durasi_Perawatan_Hari'].clip(lower=0)


### 6b. Feature Engineering Lanjutan (Claim-Level)
Berdasarkan adaptasi pipeline *Best Practice* asuransi kesehatan, fitur tambahan untuk level-klaim dibuat untuk memperkaya sinyal prediktif.

In [ ]:
# 1. Usia pasien saat klaim (tahun)
df_master['Usia_Saat_Klaim'] = (df_master['Tanggal Pasien Masuk RS'] - df_master['Tanggal Lahir']).dt.days / 365.25

# 2. Lama polis aktif saat klaim (tahun)
df_master['Lama_Polis_Aktif'] = (df_master['Tanggal Pasien Masuk RS'] - df_master['Tanggal Efektif Polis']).dt.days / 365.25

# 3. Rasio klaim (Approved / Billed) — WINSORIZE P1-P99
df_master['Rasio_Klaim'] = df_master['Nominal Klaim Yang Disetujui'] / df_master['Nominal Biaya RS Yang Terjadi'].replace(0, np.nan)
p1  = df_master['Rasio_Klaim'].quantile(0.01)
p99 = df_master['Rasio_Klaim'].quantile(0.99)
df_master['Rasio_Klaim'] = df_master['Rasio_Klaim'].clip(lower=p1, upper=p99)
df_master['Rasio_Klaim'] = df_master['Rasio_Klaim'].fillna(df_master['Rasio_Klaim'].median())

# 4. Selisih Nominal
df_master['Selisih_Klaim'] = df_master['Nominal Klaim Yang Disetujui'] - df_master['Nominal Biaya RS Yang Terjadi'].fillna(0)

# 5. Kategori ICD (huruf pertama)
df_master['ICD_Kategori'] = df_master['ICD Diagnosis'].astype(str).str[0]

# 6. Binary flags
df_master['is_inpatient'] = (df_master['Inpatient/Outpatient'].str.strip() == 'IP').astype(int)
df_master['is_cashless']  = (df_master['Reimburse/Cashless'].str.strip().str.upper() == 'C').astype(int)
df_master['is_overseas']  = (~df_master['Lokasi RS'].str.contains('Indonesia', case=False, na=False)).astype(int)
df_master['is_singapore'] = (df_master['Lokasi RS'] == 'Singapore').astype(int)

# 7. Log nominal untuk meratakan distribusi (skewness handling)
df_master['log_nominal_disetujui'] = np.log1p(df_master['Nominal Klaim Yang Disetujui'])


### 6c. Konstruksi Time-Series Bulanan & Agregasi Historis
Target kompetisi adalah memprediksi **Claim Frequency, Total Claim, dan Claim Severity bulanan**. Kita perlu meng-roll-up data klaim ke tingkat bulan.

In [ ]:
# Ekstrak Tahun_Bulan untuk agregasi
df_master['year_month'] = df_master['Tanggal Pasien Masuk RS'].dt.to_period('M')

# Agregasi Target Utama Bulanan
monthly_targets = df_master.groupby('year_month').agg(
    Claim_Frequency  = ('Claim ID', 'count'),
    Total_Claim      = ('Nominal_Klaim_Capped', 'sum'),
).reset_index()
# Hitung ulang Claim Severity dari Total Claim yang baru (Capped)
monthly_targets['Claim_Severity'] = monthly_targets['Total_Claim'] / monthly_targets['Claim_Frequency']

# Agregasi Fitur Pendukung (Feature Engineering Temporal)
monthly_feats = df_master.groupby('year_month').agg(
    avg_age          = ('Usia_Saat_Klaim', 'mean'),
    avg_los          = ('Durasi_Perawatan_Hari', 'mean'),
    avg_rasio        = ('Rasio_Klaim', 'mean'),
    pct_inpatient    = ('is_inpatient', 'mean'),
    pct_cashless     = ('is_cashless', 'mean'),
    pct_overseas     = ('is_overseas', 'mean'),
    unique_polis     = ('Nomor Polis', 'nunique'),
    avg_tenure       = ('Lama_Polis_Aktif', 'mean'),
).reset_index()

# Join menjadi df_monthly
df_monthly = monthly_targets.merge(monthly_feats, on='year_month', how='left')

# Konversi 'year_month' ke timestamp
df_monthly['date'] = df_monthly['year_month'].dt.to_timestamp()
df_monthly = df_monthly.sort_values('date').reset_index(drop=True)

# ==========================================
# PENAMBAHAN TIME-WINDOW FEATURES
# ==========================================
targets = ['Claim_Frequency', 'Total_Claim', 'Claim_Severity']
for tgt in targets:
    # Lag Features (Mundur 1, 2, 3 bulan)
    for lag_n in [1, 2, 3]:
        df_monthly[f'{tgt}_Lag{lag_n}'] = df_monthly[tgt].shift(lag_n)
    
    # Rolling Average 3M (menggunakan shift(1) agar terhindar dari data leakage)
    df_monthly[f'{tgt}_RollAvg_3M'] = df_monthly[tgt].shift(1).rolling(window=3).mean()


### 7. Verifikasi Hasil Akhir

In [ ]:
print("=== INFO DATA MASTER ===")
df_master.info()

print("\n=== IDENTIFIKASI MISSING VALUES ===")
print(df_master.isnull().sum())

print("\n=== BENTUK DATA ===")
print(f"Total Baris: {df_master.shape[0]}, Total Kolom: {df_master.shape[1]}")

df_master.head()

### 8. Rangkuman Data Preprocessing & Ekspor Dataset
Berikut merupakan ringkasan dari serangkaian langkah Data Cleaning & Feature Engineering yang diimplementasikan:
- **Pembersihan Logika Kas**: Menghilangkan baris kosong kas (Disetujui = 0), dan menyamakan Biaya RS yang = 0 padahal ada persetujuan yang diklaim (Imputasi Human Error).
- **Validasi Outlier (Capping/Winsorize)**: Membangun kolom `Nominal_Klaim_Capped` dengan memotong angka klaim tinggi di batas P99. Rasio tagihan juga di-*winsorize* (P1–P99) menghilangkan efek outlier rasio ekstrem.
- **Feature Extraction (Micro Level)**: Melahirkan banyak sinyal demografis/finansial kuat berupa Usia Saat Rawat, Tenure Polis, Durasi Inap (LOS), `Selisih_Klaim`, dan berbagai Binomial flags (Cashless, Inpatient, dsb).
- **Rollup Time-Series (Macro Level)**: Data individu `df_master` diagregasikan ke format bulan per kejadian (*Incurred Basis*) membentuk kerangka `df_monthly` untuk target (Frequency, Severity, Total Claim) lengkap beserta turunan fitur di atasnya.

**Ekspor Dataset**
Kedua bentuk (*granular master* dan *rolled-up time series*) disaving ke CSV untuk stage model training.

In [ ]:
import os
output_dir = r"d:\grinding\Lomba\MCF ITB\yokk juara"

df_master.to_csv(os.path.join(output_dir, 'Data_Master_Cleaned.csv'), index=False)
df_monthly.to_csv(os.path.join(output_dir, 'Data_Monthly_TimeSeries.csv'), index=False)

print("✅ Ekspor Selesai:")
print(f"1. Data Master (Klaim Demografi Detail): {df_master.shape[0]} baris, {df_master.shape[1]} kolom")
print(f"2. Data Bulanan (Time-Series): {df_monthly.shape[0]} bulan, {df_monthly.shape[1]} kolom/fitur")
